# 🚀 3D Scene Understanding — Google Colab GPU Server (`scene-compute`)

Notebook này biến Google Colab thành GPU server từ xa cho hệ thống **3D Scene Understanding**.

### Cách hoạt động:
1. Chạy notebook này trên Google Colab với GPU (T4 / L4 / A100).
2. Cloudflare Tunnel tạo một URL HTTPS công khai (ví dụ `https://xxxx.trycloudflare.com`).
3. Điền URL này vào `COLAB_URL` trong file `backend/.env` trên máy local của bạn.
4. Khi bạn upload ảnh và xử lý, máy local sẽ tự động gửi tác vụ GPU (Depth Anything 3, SAM 3, GeoCalib, MetaCLIP 2) sang Colab!

### Bước 1: Kiểm tra GPU
*(Hãy chắc chắn bạn đã vào menu **Runtime** → **Change runtime type** → chọn **T4 GPU** hoặc cao hơn)*

In [ ]:
!nvidia-smi

Fri Oct 16 09:37:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Bước 2: Cài đặt `uv` và Python 3.12

In [ ]:
!pip install -q uv
!uv python install 3.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 123.4 MB/s eta 0:00:00
Installed Python 3.12.15 in 930ms
 + cpython-3.12.15-linux-x86_64-gnu (python3.12)


### Bước 3: Tải mã nguồn dự án
*(Thay đổi URL git bên dưới thành repository của bạn, hoặc upload file zip mã nguồn lên Colab)*

In [ ]:
# Thay bằng repo GitHub của bạn
REPO_URL = "https://github.com/luatducvo/3D-scene.git"

import os
if not os.path.exists("/content/3D-scene"):
    !git clone $REPO_URL /content/3D-scene

%cd /content/3D-scene

Cloning into '/content/3D-scene'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (554/554), done.
remote: Compressing objects: 100% (338/338), done.
remote: Total 554 (delta 210), reused 500 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (554/554), 923.49 KiB | 18.11 MiB/s, done.
Resolving deltas: 100% (210/210), done.
/content/3D-scene


### Bước 4: Cài đặt thư viện phụ thuộc cho `scene-compute`
- Chạy với `--extra gpu` để cài đặt đầy đủ Depth Anything 3, SAM 3, GeoCalib, MetaCLIP 2.

In [ ]:
# Cài đặt GPU packages
!uv sync --python 3.12 --package scene-compute --extra gpu

Using CPython 3.12.15
Creating virtual environment at: .venv
Resolved 184 packages in 2ms
Prepared 169 packages in 27.20s
Installed 169 packages in 272ms
 + addict==2.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + apsw==3.54.0.0
 + argcomplete==3.7.2
 + asttokens==3.0.2
 + attrs==26.1.0
 + backports-zstd==1.8.0
 + blinker==1.9.0
 + certifi==2026.7.22
 + cfgv==3.5.0
 + charset-normalizer==3.5.2
 + click==8.5.0
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40)
 + colorama==0.4.6
 + comm==0.2.3
 + configargparse==1.8.0
 + contourpy==1.3.3
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + dash==4.4.1
 + decorator==4.4.2
 + depth-anything-3==0.0.0 (from git+https://github.com/ByteDance-Seed/Depth-Anything-3.git@3d835ec1a5802d64a8b8b15f817a1ab54809bfe4)
 + distlib==0.4.3
 + e3nn==0.6.0
 + einops==0.8.2
 + evo==1.38.0
 + executing==

### Bước 5: (Tùy chọn) Tải trước model weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR = "/content/drive/MyDrive/model-weights"
MODELS_DIR = f"{DRIVE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Thiết lập biến môi trường trỏ vào Google Drive
os.environ["DATA_DIR"] = "/content/data"        # Data scene tạm để ở SSD Colab cho nhanh
os.environ["MODELS_DIR"] = MODELS_DIR          # Weights lưu vĩnh viễn trên Drive
os.environ["HF_HOME"] = f"{MODELS_DIR}/hf"
os.environ["TORCH_HOME"] = f"{MODELS_DIR}/torch"
os.environ["MODELS_PIPELINE_CONFIG"] = "/content/3D-scene/backend/config/pipeline.yaml"

# Nếu bạn dùng SAM 3 gated weights từ HuggingFace, bỏ comment dòng dưới:
os.environ["HF_TOKEN"] = ""

!uv run --python 3.12 python -m scene_compute.download_models

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installed 13 packages in 9ms
download ingest.opencv
download recon.da3





download align.geocalib_ransac

download mesh.tsdf
download segment.sam3

download lift.maskclustering
download embed.metaclip2












download attributes.none
download graph.geometric


### Bước 6: Khởi chạy Cloudflare Tunnel
Tạo đường hầm HTTPS an toàn để kết nối từ máy tính của bạn về Colab.

In [ ]:
import subprocess, time, re

# Cài đặt cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Chạy tunnel nền chuyển tiếp port 9000
!pkill cloudflared || true
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:9001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("=" * 60)
print("🎉 URL PUBLIC CHO MÁY LOCAL:", public_url)
print("=" * 60)
print("👉 Copy URL trên dán vào `COLAB_URL` trong file `backend/.env` ở máy local của bạn!")

🎉 URL PUBLIC CHO MÁY LOCAL: https://intellectual-compromise-investors-beatles.trycloudflare.com
👉 Copy URL trên dán vào `COLAB_URL` trong file `backend/.env` ở máy local của bạn!


### Bước 7: Khởi chạy `scene-compute` HTTP Server
*(Giữ cell này luôn chạy trong suốt quá trình bạn thao tác trên web app)*

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/model-weights"
MODELS_DIR = f"{DRIVE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Bạn có thể đặt mật khẩu token bất kỳ (nhớ điền khớp vào COMPUTE_TOKEN trong backend/.env)
COMPUTE_TOKEN = ""

os.environ["COMPUTE_TOKEN"] = COMPUTE_TOKEN
os.environ["PORT"] = "9000"
os.environ["DATA_DIR"] = "/content/data"
os.environ["MODELS_DIR"] = MODELS_DIR
os.environ["HF_HOME"] = f"{MODELS_DIR}/hf"
os.environ["TORCH_HOME"] = f"{MODELS_DIR}/torch"
os.environ["CHUNK_SIZE_MB"] = "25"

print(f"Starting scene-compute on port 9000 with token: {COMPUTE_TOKEN}")
!uv run --python 3.12 uvicorn scene_compute.server:app --host 0.0.0.0 --port 9001

Starting scene-compute on port 9000 with token: fQQ9yhPbtUWMEj8plTGyNTx148AIeiibF2IdxEnjVLY
INFO:     Started server process [3939]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9001 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3939]
